In [ ]:
import SimpleITK as sitk
import numpy as np
import os
import math
import matplotlib.pyplot as plt
from ipywidgets import interact, fixed
from IPython.display import clear_output

M_UM_SCALE = 1000000
downsample = 64


In [ ]:

def _validate_landmarks(
    fixed_landmarks,
    moving_landmarks,
    min_landmarks=4,
):
    """
    Validate corresponding 3-D physical-space landmarks.

    Returns
    -------
    fixed : np.ndarray, shape (N, 3)
    moving : np.ndarray, shape (N, 3)
    """
    if fixed_landmarks is None or moving_landmarks is None:
        raise ValueError(
            "Both fixed_landmarks and moving_landmarks are required."
        )

    fixed = np.asarray(fixed_landmarks, dtype=np.float64)
    moving = np.asarray(moving_landmarks, dtype=np.float64)

    if fixed.ndim != 2 or fixed.shape[1] != 3:
        raise ValueError(
            f"fixed_landmarks must have shape (N, 3), got {fixed.shape}"
        )
    if moving.ndim != 2 or moving.shape[1] != 3:
        raise ValueError(
            f"moving_landmarks must have shape (N, 3), got {moving.shape}"
        )
    if fixed.shape != moving.shape:
        raise ValueError(
            "fixed_landmarks and moving_landmarks must have identical shapes: "
            f"{fixed.shape} != {moving.shape}"
        )
    if fixed.shape[0] < min_landmarks:
        raise ValueError(
            f"At least {min_landmarks} corresponding 3-D landmarks are required; "
            f"got {fixed.shape[0]}"
        )
    if not np.isfinite(fixed).all() or not np.isfinite(moving).all():
        raise ValueError("Landmarks must contain only finite values.")

    # Reject duplicate/near-duplicate points because they make an affine
    # initialization poorly conditioned.
    if fixed.shape[0] > 1:
        fixed_dist = np.linalg.norm(
            fixed[:, None, :] - fixed[None, :, :], axis=2
        )
        moving_dist = np.linalg.norm(
            moving[:, None, :] - moving[None, :, :], axis=2
        )
        fixed_nonzero = fixed_dist[np.triu_indices(fixed.shape[0], k=1)]
        moving_nonzero = moving_dist[np.triu_indices(moving.shape[0], k=1)]

        if np.any(fixed_nonzero <= 1e-6):
            raise ValueError("Fixed landmarks contain duplicate/near-duplicate points.")
        if np.any(moving_nonzero <= 1e-6):
            raise ValueError("Moving landmarks contain duplicate/near-duplicate points.")

    return fixed, moving


def _landmark_affine(
    fixed_landmarks,
    moving_landmarks,
):
    """
    Create a SimpleITK affine transform initialized from corresponding
    physical-space landmarks.

    The resulting transform follows the same convention used by
    sitk.Resample: it maps fixed/output physical coordinates to
    moving/input coordinates.
    """
    fixed, moving = _validate_landmarks(
        fixed_landmarks,
        moving_landmarks,
    )

    transform = sitk.AffineTransform(3)

    transform = sitk.LandmarkBasedTransformInitializer(
        transform,
        fixed.flatten().tolist(),
        moving.flatten().tolist(),
    )

    return transform


def _landmark_residuals(
    transform,
    fixed_landmarks,
    moving_landmarks,
):
    """
    Calculate landmark residuals for a transform.

    Because SimpleITK's resampling convention is output/fixed -> input/moving,
    fixed landmarks are transformed and compared with moving landmarks.
    """
    fixed, moving = _validate_landmarks(
        fixed_landmarks,
        moving_landmarks,
    )

    #predicted = np.asarray([transform.TransformPoint(tuple(p)) for p in fixed],dtype=np.float64,)
    #residual_vectors = predicted - moving
    predicted = np.asarray([transform.GetInverse().TransformPoint(tuple(p)) for p in moving],dtype=np.float64,)
    residual_vectors = predicted - fixed
    residuals = np.linalg.norm(residual_vectors, axis=1)

    return residuals, residual_vectors


def _landmark_report(
    transform,
    fixed_landmarks,
    moving_landmarks,
    names=None,
    report_title="put something here"
):
    """
    Print and return landmark residual statistics.
    """
    fixed, moving = _validate_landmarks(
        fixed_landmarks,
        moving_landmarks,
    )

    residuals, residual_vectors = _landmark_residuals(
        transform,
        fixed,
        moving,
    )

    #distances = np.linalg.norm(fixed-predicted)
    #print(distances)

    if names is None:
        names = [f"landmark_{i:02d}" for i in range(len(residuals))]
    if len(names) != len(residuals):
        raise ValueError("Landmark names must have the same length as landmarks.")

    rms = float(np.sqrt(np.mean(residuals ** 2)))
    mean = float(np.mean(residuals))
    median = float(np.median(residuals))
    maximum = float(np.max(residuals))

    print(f"\nLandmark residuals {report_title}:")
    for name, residual in zip(names, residuals):
        print(f"  {name:24s}: {residual:10.3f} µm")

    print(
        f"Landmark RMS={rms:.3f} µm, "
        f"mean={mean:.3f} µm, "
        f"median={median:.3f} µm, "
        f"max={maximum:.3f} µm"
    )

    return {
        "rms": rms,
        "mean": mean,
        "median": median,
        "max": maximum,
        "residuals": residuals.tolist(),
        "residual_vectors": residual_vectors.tolist(),
        "names": list(names),
    }


def _load_landmarks(data):


    if isinstance(data, dict):
        entries = data.get("landmarks")
    elif isinstance(data, list):
        entries = data
    else:
        raise ValueError(
            "Landmark JSON must be either an object containing "
            "'landmarks' or a list of landmark objects."
        )

    if not isinstance(entries, list) or not entries:
        raise ValueError("No landmarks were found in the landmark JSON.")

    fixed = []
    moving = []
    names = []

    for i, item in enumerate(entries):
        if not isinstance(item, dict):
            raise ValueError(f"Landmark {i} must be an object.")

        if "fixed" not in item or "moving" not in item:
            raise ValueError(
                f"Landmark {i} must contain both 'fixed' and 'moving'."
            )

        name = str(item.get("name", f"landmark_{i:02d}"))
        fixed_point = item["fixed"]
        moving_point = item["moving"]

        if len(fixed_point) != 3 or len(moving_point) != 3:
            raise ValueError(
                f"Landmark '{name}' must contain 3-D fixed and moving coordinates."
            )

        fixed.append(fixed_point)
        moving.append(moving_point)
        names.append(name)

    fixed, moving = _validate_landmarks(
        fixed,
        moving,
    )

    return {
        "fixed": fixed,
        "moving": moving,
        "names": names,
    }


def index_to_physical_point(image, index_zyx):
    """
    Convert a numpy-style (z, y, x) voxel index to SimpleITK physical
    coordinates (x, y, z).
    """
    if len(index_zyx) != 3:
        raise ValueError("index_zyx must contain exactly three values.")

    z, y, x = [int(v) for v in index_zyx]
    return image.TransformIndexToPhysicalPoint((x, y, z))


def physical_to_index_zyx(image, point_xyz):
    """
    Convert SimpleITK physical coordinates (x, y, z) to numpy-style
    (z, y, x) continuous index coordinates.
    """
    if len(point_xyz) != 3:
        raise ValueError("point_xyz must contain exactly three values.")

    continuous_index = image.TransformPhysicalPointToContinuousIndex(
        tuple(float(v) for v in point_xyz)
    )
    x, y, z = continuous_index
    return z, y, x

# Callback invoked when the StartEvent happens, sets up our new data.
def start_plot():
    global metric_values, multires_iterations

    metric_values = []
    multires_iterations = []


# Callback invoked when the EndEvent happens, do cleanup of data and figure.
def end_plot():
    global metric_values, multires_iterations

    del metric_values
    del multires_iterations
    # Close figure, we don't want to get a duplicate of the plot latter on.
    plt.close()


# Callback invoked when the IterationEvent happens, update our data and display new figure.
def plot_values(registration):
    global metric_values, multires_iterations

    metric_values.append(registration.GetMetricValue())
    # Clear the output area (wait=True, to reduce flickering), and plot current data
    clear_output(wait=True)
    # Plot the similarity metric values
    plt.plot(metric_values, "r")
    plt.plot(
        multires_iterations,
        [metric_values[index] for index in multires_iterations],
        "b*",
    )
    plt.xlabel("Iteration Number", fontsize=12)
    plt.ylabel("Metric Value", fontsize=12)
    plt.show()


# Callback invoked when the sitkMultiResolutionIterationEvent happens, update the index into the
# metric_values list.
def update_multires_iterations():
    global metric_values, multires_iterations
    multires_iterations.append(len(metric_values))

In [ ]:
def get_points_from_db(brain):
    points = {}
    points['DK55'] = [
        [0.012119069695472717, 0.006376158911734819, 0.0035099999513477087],
        [0.012223436497151852, 0.006139863282442093, 0.0063299997709691525],
        [0.01276596449315548, 0.006230868399143219, 0.004530000034719706],
        [0.012741747312247753, 0.006207763217389584, 0.005271818023175001],
        [0.012955991551280022, 0.007423400413244963, 0.0037899999879300594],
        [0.013092001900076866, 0.007213350851088762, 0.0062699997797608376],
        [0.01262159738689661, 0.005424220114946365, 0.003930000122636557],
        [0.012653084471821785, 0.005337724927812815, 0.005710000172257423]        
    ]
    points['Allen'] = [
        [0.01019303, 0.00528655, 0.00409163],
        [0.01019342, 0.00528553, 0.00729279],
        [0.01077123, 0.005214760000000001, 0.00529086],
        [0.01077144, 0.00521459, 0.00609408],
        [0.010852700000000002, 0.00677545, 0.004339979999999999],
        [0.01085258, 0.0067753100000000005, 0.007045180000000001],
        [0.01070959, 0.00427799, 0.004731979999999999],
        [0.0107099, 0.00427852, 0.00665216]
    ]
    
    db_points = None


    try:
        db_points = points[brain]
    except KeyError:
        return None

    data = []
    for (x,y,z) in db_points:
        # Perform operation on the 3 numbers
        x *= (M_UM_SCALE)
        y *= (M_UM_SCALE)
        z *= (M_UM_SCALE)
        data.append((x,y,z))

    return data


In [ ]:
moving_brain = 'DK55'
fixed_brain = 'Allen'
moving_data = get_points_from_db(moving_brain)
fixed_data = get_points_from_db(fixed_brain)

In [ ]:
structures = ['5N_L', '5N_R','6N_L', '6N_R','7N_L','7N_R', 'LC_L','LC_R']
data = []
for s,m,f in zip(structures, moving_data, fixed_data):
    tmp = {}
    tmp["name"] = s 
    tmp["fixed"] = f
    tmp["moving"] = m
    data.append(tmp)

In [ ]:
# validation on 5N_L to match data in Neuroglancer ID=1173
xum,yum,zum = data[0]['moving']
x = xum / 10.4
y = yum / 10.4
z = zum / 20
print(x,y,z)

In [ ]:
landmark_data = _load_landmarks(data)
fixed_landmarks=landmark_data["fixed"]
moving_landmarks=landmark_data["moving"]
landmark_names=landmark_data["names"]

In [ ]:
fixed, moving = _validate_landmarks(
    fixed_landmarks,
    moving_landmarks,
)

In [ ]:
def affine_registration(fixed_image,moving_image, fixed_landmarks=None,
        moving_landmarks=None,
        landmark_names=None,
):

    if fixed_image.GetDimension() != 3 or moving_image.GetDimension() != 3:
        raise ValueError("affine_registration requires 3-D images.")

    if fixed_image.GetNumberOfComponentsPerPixel() == 3:
        fixed_image = sitk.VectorIndexSelectionCast(fixed_image, 1)
    if moving_image.GetNumberOfComponentsPerPixel() == 3:
        moving_image = sitk.VectorIndexSelectionCast(moving_image, 1)

    fixed_image = sitk.Cast(fixed_image, sitk.sitkFloat32)
    moving_image = sitk.Cast(moving_image, sitk.sitkFloat32)

    # The registration pipeline uses a common zero-origin Cartesian
    # coordinate system. Landmark coordinates must use this same system.
    fixed_image.SetOrigin((0.0, 0.0, 0.0))
    moving_image.SetOrigin((0.0, 0.0, 0.0))
    fixed_image.SetDirection(np.eye(3).flatten())
    moving_image.SetDirection(np.eye(3).flatten())

    print(f"Affine registration with fixed image size {fixed_image.GetSize()} and moving image size {moving_image.GetSize()}")
    print(f"Fixed image spacing: {fixed_image.GetSpacing()}")
    print(f"Moving image spacing: {moving_image.GetSpacing()}")
    initial_transform = sitk.CenteredTransformInitializer(
        fixed_image,
        moving_image,
        sitk.AffineTransform(3),
        sitk.CenteredTransformInitializerFilter.GEOMETRY)
    registration = sitk.ImageRegistrationMethod()
    registration.SetInitialTransform(sitk.AffineTransform(initial_transform), inPlace=False)
    # initial preview image
    
    #registration.SetMetricAsCorrelation()
    registration.SetMetricAsJointHistogramMutualInformation()

    registration.SetMetricSamplingStrategy(registration.RANDOM)
    registration.SetMetricSamplingPercentage(0.01)
    # Optimizer settings.
    registration.SetOptimizerAsGradientDescent(
        learningRate=1,
        numberOfIterations=300,
        convergenceMinimumValue=1e-6,
        convergenceWindowSize=10
    )    
    # --- Setup Metric, Optimizer, & Interpolator ---
    registration.SetOptimizerScalesFromPhysicalShift()
    registration.SetInterpolator(sitk.sitkLinear)    
    # --- Multi-Resolution ---
    registration.SetShrinkFactorsPerLevel(shrinkFactors=[4, 2, 1])
    registration.SetSmoothingSigmasPerLevel(smoothingSigmas=[2, 1, 0])
    # Connect all of the observers so that we can perform plotting during registration.
    registration.AddCommand(sitk.sitkStartEvent, start_plot)
    registration.AddCommand(sitk.sitkEndEvent, end_plot)
    registration.AddCommand(sitk.sitkMultiResolutionIterationEvent, update_multires_iterations)
    registration.AddCommand(sitk.sitkIterationEvent, lambda: plot_values(registration))
    
    # Execute and Resample
    final_transform = registration.Execute(fixed_image, moving_image)    

    print("\nAffine registration complete.")
    print(f"Final metric: {registration.GetMetricValue()}")
    print("Optimizer stopping condition: ", registration.GetOptimizerStopConditionDescription())

    return final_transform


In [ ]:
# Paths to fixed and moving 3D images
reg_path = '/net/birdstore/Active_Atlas_Data/data_root/brains_info/registration'
fixed_image_path = os.path.join(reg_path, fixed_brain, f'source.{downsample}.nii')
moving_image_path = os.path.join(reg_path, moving_brain, f'source.{downsample}.nii')
fixed_image = sitk.ReadImage(fixed_image_path, sitk.sitkFloat32)
moving_image = sitk.ReadImage(moving_image_path, sitk.sitkFloat32)
print(f'moving image spacing {moving_image.GetSpacing()}')
fixed_image.SetSpacing((20.0, 20.0, 10.0))
print(f'fixed image spacing {fixed_image.GetSpacing()}')


In [ ]:
%%time
affine_transform = affine_registration(fixed_image,moving_image,fixed_landmarks=landmark_data["fixed"],
                moving_landmarks=landmark_data["moving"],
                landmark_names=landmark_data["names"],
)

In [ ]:
moving_resampled = sitk.Resample(
    moving_image,
    fixed_image,
    affine_transform,
    sitk.sitkLinear,
    0.0,
    moving_image.GetPixelID(),
)
output_path = os.path.join(reg_path, moving_brain, f'notebook.reg.{downsample}.nii')
resultImage = sitk.Cast(sitk.RescaleIntensity(moving_resampled), sitk.sitkUInt16)
sitk.WriteImage(resultImage, output_path)


In [ ]:
transform_path = os.path.join(reg_path, f'notebook_{moving_brain}_{fixed_brain}.{downsample}.tfm')
sitk.WriteTransform(affine_transform, transform_path)

In [ ]:
affine_transform = sitk.ReadTransform("/net/birdstore/Active_Atlas_Data/data_root/brains_info/registration/MD585_DK55.tfm")

In [ ]:
moving_point = data[-1]['moving']
moving_to_fixed_transform = affine_transform.GetInverse()
moved_point = moving_to_fixed_transform.TransformPoint(moving_point)
print('moving', moving_point)
print('moved', moved_point)
fixed_point = data[-1]['fixed']
print('fixed',fixed_point)

In [ ]:
def calculate_target_registration_error(moving_points_um,fixed_points_um,affine_transform,):
    # 1. Cast the generic transform strictly to an AffineTransform and invert it
    # SimpleITK transforms map Fixed -> Moving. To map actual drawn points,
    # we must invert it to map Moving -> Fixed.
    #moving_to_fixed_transform = sitk.AffineTransform(affine_transform).GetInverse()
    moving_to_fixed_transform = affine_transform.GetInverse()

    errors_in_microns = []

    # 2. Iterate through each paired point
    for moving_um, fixed_um in zip(moving_points_um, fixed_points_um):


        # B. Apply the inverted transform (Moving Physical mm -> Fixed Physical um)
        moved = moving_to_fixed_transform.TransformPoint(moving_um)

        # C. Convert ground-truth fixed voxel index to fixed physical space (um)
        #pt_fix_phys = fixed_image.TransformIndexToPhysicalPoint([int(c) for c in pt_fix_vox])

        # D. Calculate Euclidean distance in physical space (um)
        distance_um = np.linalg.norm(np.array(moved) - np.array(fixed_um))

        errors_in_microns.append(distance_um)

    errors_array = np.array(errors_in_microns)

    # 3. Compile statistical metrics
    stats = {
        "errors_per_point": errors_array,
        "mean_tre": np.mean(errors_array),
        "median_tre": np.median(errors_array),
        "std_tre": np.std(errors_array),
        "max_tre": np.max(errors_array),
        "min_tre": np.min(errors_array),
    }

    return stats

In [ ]:
stats = calculate_target_registration_error(moving_landmarks,fixed_landmarks,affine_transform)

In [ ]:
stats

In [ ]:
moving_test = data[9]['moving']
fixed_test = data[9]['fixed']
moved = affine_transform.GetInverse().TransformPoint(moving_test)
print('moving', moving_test)
print('fixed', fixed_test)
print('moved', moved)
fixed_spacing = (0.325,0.325,20)
moving_spacing = (0.452,0.452,20)
fixed_voxels = np.array(fixed_test) / np.array(fixed_spacing)
moved_voxels = np.array(moved) / np.array(moving_spacing)
print('fixed voxels', fixed_voxels) 
print('moved voxels', moved_voxels)